# E-Commerce Customer Segmentation (RFM + K-Means)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

sns.set_theme(style='whitegrid')

# Load data
The Excel file has two sheets (Dec 2009 - Dec 2010 and Dec 2010 - Dec 2011), so both are loaded and combined.

In [ ]:
file_path = '/content/online_retail_II.xlsx'
sheets = pd.read_excel(file_path, sheet_name=None)   # read all sheets
df = pd.concat(sheets.values(), ignore_index=True)
print(df.shape)
df.head()

# Data cleaning
The two sheets overlap in December 2010, so exact duplicates are removed first.

In [ ]:
print('Raw rows:', len(df))

df['Invoice'] = df['Invoice'].astype(str)
df['StockCode'] = df['StockCode'].astype(str).str.upper().str.strip()

# 1) duplicates (the two sheets overlap)
df = df.drop_duplicates()
print('After removing duplicates:', len(df))

# 2) missing Customer ID (we can't build RFM without a customer)
before = len(df)
df = df.dropna(subset=['Customer ID'])
print(f'Removed {before - len(df):,} rows without Customer ID ({(before - len(df)) / before:.1%})')

# 3) cancelled invoices start with "C"
df = df[~df['Invoice'].str.startswith('C')]

# 4) keep only positive quantity and price
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# 5) remove non-product codes (postage, fees, manual adjustments)
non_product = ['POST', 'DOT', 'M', 'BANK CHARGES', 'AMAZONFEE', 'ADJUST', 'ADJUST2',
               'B', 'CRUK', 'D', 'S', 'C2', 'TEST001', 'TEST002', 'PADS']
df = df[~(df['StockCode'].isin(non_product) | df['StockCode'].str.startswith('GIFT'))]

df['TotalPrice'] = df['Quantity'] * df['Price']
print('Clean rows:', len(df), '| Customers:', df['Customer ID'].nunique())

# RFM feature engineering
* Recency: days since the customer's last purchase
* Frequency: number of invoices
* Monetary: total spend

In [ ]:
# day after the last date in the data (used to calculate Recency)
last_date = df['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda d: (last_date - d.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalPrice', 'sum'),
)
rfm.describe()

# Log transform + scaling
RFM data is very right-skewed (a few huge customers). K-Means uses distances, so without a log transform it wastes clusters on a handful of extreme customers (this caused a cluster of only 5 customers in my first version). `log1p` fixes this, then StandardScaler puts all three features on the same scale.

In [ ]:
cols = ['Recency', 'Frequency', 'Monetary']
rfm_log = np.log1p(rfm[cols])

# skewness before and after (closer to 0 = more symmetric)
print(pd.DataFrame({'before': rfm[cols].skew(), 'after log1p': rfm_log.skew()}).round(2))

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Choosing the number of clusters (k)
I compare k = 2 to 10 using three measures: elbow (inertia), silhouette (higher is better) and Davies-Bouldin (lower is better). I also check the size of the smallest cluster.

In [ ]:
results = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(rfm_scaled)
    results.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(rfm_scaled, km.labels_),
        'davies_bouldin': davies_bouldin_score(rfm_scaled, km.labels_),
        'smallest_cluster_%': 100 * np.bincount(km.labels_).min() / len(rfm_scaled),
    })
k_table = pd.DataFrame(results).set_index('k')
print(k_table.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(k_table.index, k_table['inertia'], 'o-');            axes[0].set_title('Elbow (inertia)')
axes[1].plot(k_table.index, k_table['silhouette'], 'o-');         axes[1].set_title('Silhouette (higher = better)')
axes[2].plot(k_table.index, k_table['davies_bouldin'], 'o-');     axes[2].set_title('Davies-Bouldin (lower = better)')
for ax in axes:
    ax.set_xlabel('k')
plt.tight_layout()
plt.show()

I limited k to 3-6 (practical for marketing), ignored any k with a cluster below 3% of customers, and picked the best silhouette among the rest, which gave **k = 4**.

# K-Means clustering

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, n_init=10, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

print(f"Silhouette Score: {silhouette_score(rfm_scaled, rfm['Cluster']):.3f}")
print(f"Davies-Bouldin:   {davies_bouldin_score(rfm_scaled, rfm['Cluster']):.3f}")

# Segment profiles
I use the **median** next to the size and revenue share, because the mean is pulled up by a few big customers.

In [ ]:
profile = rfm.groupby('Cluster').agg(
    Customers=('Recency', 'size'),
    Recency_median=('Recency', 'median'),
    Frequency_median=('Frequency', 'median'),
    Monetary_median=('Monetary', 'median'),
    Revenue=('Monetary', 'sum'),
)
profile['Customers_%'] = (100 * profile['Customers'] / profile['Customers'].sum()).round(1)
profile['Revenue_%'] = (100 * profile['Revenue'] / profile['Revenue'].sum()).round(1)
profile.drop(columns='Revenue').round(1)

# Naming the segments
I named each cluster after reading the profile table above (cluster numbers can change if the data or library version changes, so check the table first and adjust the names if needed):
* **Champions:** bought very recently, order often, highest spend
* **At Risk:** decent spend and frequency, but haven't bought for about 5 months
* **Potential Loyalists:** recent buyers with moderate frequency and spend
* **Hibernating / Lost:** one order, very long time ago, lowest spend

In [ ]:
names = {0: 'Potential Loyalists', 1: 'Champions', 2: 'Hibernating / Lost', 3: 'At Risk'}
rfm['Segment'] = rfm['Cluster'].map(names)
profile['Segment'] = profile.index.map(names)

# check that the names match the profile before using them
profile.set_index('Segment')[['Customers', 'Customers_%', 'Recency_median',
                              'Frequency_median', 'Monetary_median', 'Revenue_%']].round(1)

# Visualisations

In [ ]:
order = ['Champions', 'At Risk', 'Potential Loyalists', 'Hibernating / Lost']

# share of customers vs share of revenue
share = profile.set_index('Segment').loc[order, ['Customers_%', 'Revenue_%']]
share.plot(kind='bar', figsize=(9, 5), color=['#4c72b0', '#dd8452'])
plt.title('Share of customers vs share of revenue')
plt.ylabel('%')
plt.xlabel('')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('segments_share.png', dpi=150)
plt.show()

In [ ]:
# PCA to see the clusters in 2D
pca = PCA(n_components=2)
pca_data = pca.fit_transform(rfm_scaled)

plt.figure(figsize=(9, 6))
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=rfm['Segment'].values,
                hue_order=order, alpha=0.6, s=18)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%} of variance)')
plt.title('Customer segments (PCA of log-scaled RFM)')
plt.legend(title='Segment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('segments_pca.png', dpi=150)
plt.show()

# Limitations
* UK online gift retailer with many wholesale customers, so results may not apply to a typical shop.
* Customers without a Customer ID were removed, so the segments describe identified customers only.
* Only 3 features (RFM), and clusters overlap smoothly, so the segments are practical groups, not naturally separated ones.